In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.6 Scaling laws

> 🎯 **Goal:** Build intuition for how model size, data, and compute trade off, and see
a bigger model reach a lower loss on the same data.

**Why this matters.** Once your architecture is modern, the next question is purely
economic: given a fixed compute budget, how big should the model be and how many
tokens should it train on? Guess wrong and you waste a fortune (this is literally a
multimillion-dollar decision for frontier labs). Scaling laws are the recipe that
turns that guess into math.

**The intuition.** Empirically, loss falls in a smooth, predictable curve as you add
*parameters* (the learned weights, more of them means more capacity) or feed more
tokens (more data to learn from). The Chinchilla result added the key twist: for a
fixed amount of *compute* (the total FLOPs you can afford to spend), parameters and
tokens should grow *together*, roughly in balance. A model that is too big for its
data is undertrained and wasteful; too small and it leaves capacity on the table.

**The mechanics.** Distinguish two budgets. *Parameters* are how much the model can
store; *compute* is how much arithmetic you spend training it (it scales with
parameters times tokens times steps). Scaling laws say loss is a power-law function of
each, so doubling resources buys a predictable, diminishing drop in loss. Here we will
not derive the constants; we will just train a small model and a bigger one under
identical conditions and confirm the bigger one reaches a lower loss, the smallest
possible demonstration of the trend.

In [ ]:
from pocketlm import (CharTokenizer, PocketLM, PocketLMConfig, init_weights,
                      make_batch, estimate_loss)

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

def train_size(n_embd, n_layer, steps):
    torch.manual_seed(0)
    cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32,
                         n_layer=n_layer, n_head=2, n_embd=n_embd)
    model = init_weights(PocketLM(cfg))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(0)
    for _ in range(steps):
        x, y = make_batch(data, 32, 16, generator=g)
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    params = sum(p.numel() for p in model.parameters())
    return params, estimate_loss(model, data, 32, 16, iters=10, generator=g)

steps = 120 if os.environ.get("POCKETLM_CI") else 400
small = train_size(32, 1, steps)
big = train_size(96, 3, steps)
print(f"small: {small[0]:>6} params, loss {small[1]:.3f}")
print(f"big:   {big[0]:>6} params, loss {big[1]:.3f}")

**What just happened.** The big model (more layers, wider embeddings, hence more
parameters) reached a lower loss than the small one on the same data and the same
number of steps. That is the scaling trend in miniature: more capacity buys lower
loss, paid for with more compute. The engineering job is spending a fixed budget where
it buys the most loss reduction, and scaling laws turn that judgment into a formula.

⚠️ **Common pitfalls**
- Bigger is not free or automatic. At these toy sizes and step counts, results are
  noisy; a single run can occasionally invert. Real scaling studies average over many
  runs and many sizes to see the clean curve.
- Confusing parameters with compute. You can add parameters without training longer,
  but an undertrained giant can lose to a well-trained smaller model. That imbalance is
  exactly what Chinchilla warned about.
- Reading too much into one data point. Two sizes show a direction, not a law. The law
  is the shape of the curve across many sizes.

🏋️ **Try it yourself.** Add a third, middle-sized model (for example `n_embd=64,
n_layer=2`) and print all three losses side by side. Do they fall on a smooth
decreasing curve as parameters grow? That curve is the scaling law you are sampling.

In [ ]:
assert big[0] > small[0]                 # the big model has more parameters
import math
assert small[1] < math.log(tok.vocab_size) and big[1] < math.log(tok.vocab_size)